# Goals

Determine how the bug described in https://github.com/IQSS/dataverse.harvard.edu/issues/500 can adversley affect search results of users of Harvard Dataverse and other Dataverse repositories.

# Plan

- Get the ROR URLs and names of all organizations that depositors have entered in the Author Affiliation and Funding Information Agency fields. Use the data in my dataset at https://doi.org/10.7910/DVN/LVVOER

- For Harvard Dataverse:
    - Get the list of URLs and names of RORs in the externalvocabularyvalue table

    - Get the list of author affiliations with RORs; the list and count of datasets that have those author affiliations; and if each ROR is in the externalvocabularyvalue table
    - For each author affiliation with a ROR:
        - Search 1: Use the Search API to search for datasets by the name of the author affiliation
        - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
        - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

- For other Dataverse installations:
    - Determine which Dataverse installations have datasets with ROR URLs in their Author Affiliations or Funding Information Agencies
    - For each installation:

        - Get the list of author affiliations with RORs and the list and count of datasets that have those author affiliations (I don't know what's in each installation's externalvocabularyvalue table, so I can't include if each ROR is in the externalvocabularyvalue table)
        - For each author affiliation with a ROR:
            - Search 1: Use the Search API to search for datasets by the name of the author affiliation
            - Search 2: Use the Search API to search for datasets by the ROR URL of the author affiliation
            - Datasets returned by search of author affiliation name (Search 1) should include datasets returned by search of author affiliation ROR URL (Search 2). Indicate if that's true or false

# Import libraries and set variables

In [ ]:
import csv
import json
import os
import pandas as pd
from pathlib import Path
import platform
import re
import requests
import sys
from time import sleep
from tqdm import tqdm
tqdmBarFormat = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions
from dataverse_functions import dataverse_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

rorRegex = r'https?://ror\.org/0[a-hj-km-np-tv-z|0-9]{6}[0-9]{2}$'

# Split author_affiliations_2026.08.25-2026.09.02.csv by installation name and installation url
- Get count of datasets in each installation that have one or more author affiliation rors

In [ ]:
# Create dataframe from author_affiliations_2026.08.25-2026.09.02.csv 
authorAffiliationsDf3 = pd.read_csv(
    os.path.join('data', 'processed', 'author_affiliations_2026.08.25-2026.09.02.csv'),
    encoding='utf-8-sig')

authorAffiliationsDf3.head()

In [ ]:
# Data quality checks before counting: 

## Installation names and URLs
print(f'Rows with no installation name: {authorAffiliationsDf3["dataverse_installation_name"].isna().sum()}')
print(f'Rows with no installation URL: {authorAffiliationsDf3["dataverse_installation_url"].isna().sum()}')

## Each installation name should have only one installation URL
urlCountsDf = authorAffiliationsDf3.groupby('dataverse_installation_name')['dataverse_installation_url'].nunique()
print(f'Installation names with more than one installation URL: {(urlCountsDf > 1).sum()}')

## Dataset PIDs 
print(f'Rows with no dataset PID: {authorAffiliationsDf3["dataset_pid"].isna().sum()}')

duplicatePidRowsDf = authorAffiliationsDf3[
    authorAffiliationsDf3.duplicated(subset=['dataset_pid'], keep=False)]
duplicatePidsDf = (
    duplicatePidRowsDf
    .groupby('dataset_pid', as_index=False)
    .agg(
        row_count=('dataset_pid', 'size'),
        installation_count=('dataverse_installation_name', 'nunique'),
        installation_names=('dataverse_installation_name', lambda x: sorted(set(x.dropna()))),
        installation_urls=('dataverse_installation_url', lambda x: sorted(set(x.dropna()))))
    .sort_values('row_count', ascending=False))
print(f'Duplicated dataset_pid values in more than one installation: {(duplicatePidsDf["installation_count"] > 1).sum()}')

In [ ]:
# Get the count of datasets in each installation that have one or more author affiliation RORs.
# Count distinct dataset PIDs, since a dataset has one row per author affiliation
datasetsWithRorsDf = authorAffiliationsDf3[authorAffiliationsDf3['author_affiliation_ror'].notna()]

datasetCountsByInstallationDf = (
    datasetsWithRorsDf
    .groupby(['dataverse_installation_name', 'dataverse_installation_url'], as_index=False)
    .agg(datasets_with_ror_count=('dataset_pid', 'nunique'))
    .sort_values('datasets_with_ror_count', ascending=False)
    .reset_index(drop=True))

# Add the count of all datasets in each installation that are in authorAffiliationsDf3
totalDatasetCountsDf = (
    authorAffiliationsDf3
    .groupby('dataverse_installation_name', as_index=False)
    .agg(datasets_with_author_affiliations_count=('dataset_pid', 'nunique')))
datasetCountsByInstallationDf = datasetCountsByInstallationDf.merge(
    totalDatasetCountsDf, how='left', on='dataverse_installation_name')

print(f'Installations with at least one dataset that has an author affiliation ROR: {len(datasetCountsByInstallationDf)}')
print(f'Datasets with at least one author affiliation ROR: {datasetCountsByInstallationDf["datasets_with_ror_count"].sum()}')

datasetCountsByInstallationDf

In [ ]:
# Create a new dataframe with only the 16 installations that 
# have one or more datasets that have an author affiliation ROR

installationUrlsWithAuthorAffRors = datasetCountsByInstallationDf['dataverse_installation_url'].to_list()

authorAffiliationsOfAffRorInstallationsDf = (authorAffiliationsDf3
    .query(
        'dataverse_installation_url in @installationUrlsWithAuthorAffRors',
    engine='python'
    )
)

authorAffiliationsOfAffRorInstallationsDf.head()

## Version 1

For each installation in authorAffiliationsOfAffRorInstallationsDf:
- Create a directory
- Split authorAffiliationsOfAffRorInstallationsDf by dataverse_installation_url to create a dataframe for each installation
- Get the installation name and URL
- Export each dataframe to the installation's directory
- Get the list of distinct author affiliation names that have ROR URLs and the count of datasets with those author affiliations as names and as ROR URLs
- For each affiliation, search the installation for
  - Datasets with the author affiliation name. Get the count of datasets, stringBasedSearchCount, and the list of distinct dataset PIDs, stringBasedSearchList
  - Datasets with the ROR URL of that author affiliation. Get the count of datasets, rorUrlBasedSearchCount, and the list of distinct dataset PIDs, rorUrlBasedSearchList
  - Get count of distinct datasets in the stringBasedSearchList and the rorUrlBasedSearchList
  - Indicate if there are datasets in the stringBasedSearchList do not include all datasets in the rorUrlBasedSearchList
  - Add columns with:
    - totalDatasetCount (count of distinct datasets in the stringBasedSearchList and the rorUrlBasedSearchList)
    - stringBasedSearchCount
    - rorUrlBasedSearchCount
    - stringBasedSearchReturnsAllDatasets (True/False)

In [ ]:
countOfDistinctAuthorAffiliationNamesByInstallationDict = []
for _, installationDf in authorAffiliationsOfAffRorInstallationsDf.groupby(['dataverse_installation_url']):
    installationDf.reset_index(drop=True)

    # Get the installation name for the installationDf
    installation_name = installationDf['dataverse_installation_name'].iloc[0]
    installation_url = installationDf['dataverse_installation_url'].iloc[0]

    # Make the installation name safe for use in a filename by replacing spaces and special characters with underscores
    safe_installation_name = re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]', '_', installation_name).strip('_'))

    # Count the distinct datasets that have each author affiliation name without a ROR URL
    noRorCountsDf = (
        installationDf[installationDf['author_affiliation_ror'].isna()]
        .groupby('author_affiliation_name', as_index=False)
        .agg(count_of_datasets_with_only_name=('dataset_pid', 'nunique')))

    # Count the distinct datasets that have each author affiliation name with a ROR URL
    # and list the distinct ROR URLs of each name (a name can have more than one ROR URL)
    withRorCountsDf = (
        installationDf[installationDf['author_affiliation_ror'].notna()]
        .groupby('author_affiliation_name', as_index=False)
        .agg(
            ror_urls=('author_affiliation_ror', lambda x: sorted(set(x))),
            count_of_datasets_with_ror=('dataset_pid', 'nunique')))

    # One row per distinct author affiliation name. Names that never appear with (or without) a ROR URL get a count of 0
    affiliationCountsDf = noRorCountsDf.merge(
        withRorCountsDf, how='outer', on='author_affiliation_name')
    affiliationCountsDf[['count_of_datasets_with_only_name', 'count_of_datasets_with_ror']] = (
        affiliationCountsDf[['count_of_datasets_with_only_name', 'count_of_datasets_with_ror']]
        .fillna(0).astype(int))

    # Order the columns
    affiliationCountsDf = affiliationCountsDf[[
        'author_affiliation_name', 'ror_urls',
        'count_of_datasets_with_only_name', 'count_of_datasets_with_ror']]

    # From affiliationCountsDf remove rows where count_of_datasets_with_only_name 
    # and count_of_datasets_with_ror are 0
    affiliationCountsDf = (affiliationCountsDf
        .query(
            'count_of_datasets_with_only_name > 0 and '
            'count_of_datasets_with_ror > 0',
        engine='python'
        ))

    # Sort by count_of_datasets_with_only_name desc and reindex
    affiliationCountsDf = (
        affiliationCountsDf
        .sort_values('count_of_datasets_with_only_name', ascending=False)
        .reset_index(drop=True))

    distinctAffiliationsCount = len(affiliationCountsDf)

    if distinctAffiliationsCount > 0:
        
        # Create directory for the installation
        installationDirPath = os.path.join(desktopPath, 'installations', safe_installation_name)
        os.makedirs(installationDirPath)

        #Export the installationDf dataframe
        fileName =f'{safe_installation_name}.csv'
        filePath = os.path.join(installationDirPath , fileName)
        installationDf.to_csv(filePath, index=False, encoding='utf-8-sig')

        # Export affiliationCountsDf
        fileName = f'author_affiliation_dataset_count.csv'
        filePath = os.path.join(installationDirPath, fileName)
        affiliationCountsDf.to_csv(filePath, index=False,  encoding='utf-8-sig')

        newRow = {
            'installation_name': safe_installation_name,
            'distinct_affiliation_name_count': len(affiliationCountsDf)
        }
        countOfDistinctAuthorAffiliationNamesByInstallationDict.append(newRow)

summaryDf = pd.DataFrame(countOfDistinctAuthorAffiliationNamesByInstallationDict)
summaryDf.head()

## Version 2

In [ ]:
countOfDistinctAuthorAffiliationNamesByInstallationDict = []
for _, installationDf in authorAffiliationsOfAffRorInstallationsDf.groupby(['dataverse_installation_url']):
    installationDf.reset_index(drop=True)

    # Get the installation name for the installationDf
    installation_name = installationDf['dataverse_installation_name'].iloc[0]
    installation_url = installationDf['dataverse_installation_url'].iloc[0]

    # Make the installation name safe for use in a filename by replacing spaces and special characters with underscores
    safe_installation_name = re.sub(r'_+', '_', re.sub(r'[^a-zA-Z0-9]', '_', installation_name).strip('_'))

    # Get list of author affiliations where there's a ROR URL but no affiliation name
    query_expr = """
        (expanded_value_key_present == False and \
        author_affiliation_ror.notnull()) \
        or \
        (expanded_value_key_present == True and \
        author_affiliation_name.isnull())
    """

    # if installation_name == 'DANS Data Station Social Sciences and Humanities':
    # if installation_name == 'Harvard Dataverse':

    rorUrlPrenset_affNameNotPresentDf = (installationDf
        .query(query_expr, engine='python')
        [['author_affiliation_ror', 'ror_display_name_2026_11_01', 'expanded_value_key_present']]
        .drop_duplicates()
        .groupby(['ror_display_name_2026_11_01', 'expanded_value_key_present'])['author_affiliation_ror'].agg(list)
        .reset_index()
        [['ror_display_name_2026_11_01', 'author_affiliation_ror', 'expanded_value_key_present']]
    )

    newRow = {
        'installation_name': safe_installation_name,
        'distinct_affiliation_name_count': len(rorUrlPrenset_affNameNotPresentDf)
    }
    countOfDistinctAuthorAffiliationNamesByInstallationDict.append(newRow)

    if len(rorUrlPrenset_affNameNotPresentDf) > 0:
        
        # Create directory for the installation
        installationDirPath = os.path.join(desktopPath, 'installations_2', safe_installation_name)
        os.makedirs(installationDirPath)

        #Export the installationDf dataframe
        fileName =f'{safe_installation_name}.csv'
        filePath = os.path.join(installationDirPath , fileName)
        installationDf.to_csv(filePath, index=False, encoding='utf-8-sig')

        # Export rorUrlPrenset_affNameNotPresentDf
        fileName = f'rorUrlPrenset_affNameNotPresentDf.csv'
        filePath = os.path.join(installationDirPath, fileName)
        rorUrlPrenset_affNameNotPresentDf.to_csv(filePath, index=False,  encoding='utf-8-sig')

summaryDf = pd.DataFrame(countOfDistinctAuthorAffiliationNamesByInstallationDict)

summaryDf = (summaryDf
    .sort_values('distinct_affiliation_name_count', ascending=False)
    .reset_index(drop=True))

fileName = f'summaryDf.csv'
filePath = os.path.join(desktopPath, 'installations_2', fileName)
summaryDf.to_csv(filePath, index=False,  encoding='utf-8-sig')

summaryDf.head()

# Measure bug's affect on findability of datasets in Dataverse installations

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in each Dataverse installation